In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 日本語フォント設定
# plt.rcParams["font.family"] = "DejaVu Sans"
sns.set_style("whitegrid")
plt.style.use("default")

In [ ]:
# 日本語フォント設定
matplotlib.rc("font", family="IPAexGothic")

In [ ]:
# 現在の最大表示列数の出力
pd.get_option("display.max_columns")

In [ ]:
# 最大表示列数の指定（ここでは150列を指定）
pd.set_option("display.max_columns", 150)

In [ ]:
# データの読み込み
results_df = pd.read_csv("data/results.csv")
results_df.head()

In [ ]:
# "年" "月" "日" 列から日付列を作成
# 年,月,日を結合して日付型に変換
results_df.insert(
    0,
    "開催日",
    pd.to_datetime(
        results_df["年"].astype(str)
        + "-"
        + results_df["月"].astype(str)
        + "-"
        + results_df["日"].astype(str)
    ),
)

In [ ]:
# "年" "月" "日" "レース場番号" "レース番号"からレースID列を作成
results_df.insert(
    1,
    "レースID",
    results_df["年"].astype(str)
    + results_df["月"].astype(str).str.zfill(2)
    + results_df["日"].astype(str).str.zfill(2)
    + results_df["レース場番号"].astype(str).str.zfill(2)
    + results_df["レース番号"].astype(str).str.zfill(2),
)

In [ ]:
# "年" "月" "日" 列の削除
results_df = results_df.drop(columns=["年", "月", "日"])

In [ ]:
# 不要カラムを削除
import re

drop_columns = [
    "複勝1着_艇番",
    "複勝1着_払戻金",
    "複勝2着_艇番",
    "複勝2着_払戻金",
    "2連単_艇番",
    "2連単_払戻金",
    "2連単_人気",
    "2連複_艇番",
    "2連複_払戻金",
    "2連複_人気",
    "拡連複1_艇番",
    "拡連複1_払戻金",
    "拡連複1_人気",
    "拡連複2_艇番",
    "拡連複2_払戻金",
    "拡連複2_人気",
    "拡連複3_艇番",
    "拡連複3_払戻金",
    "拡連複3_人気",
    "3連単_艇番",
    "3連単_払戻金",
    "3連単_人気",
    "3連複_艇番",
    "3連複_払戻金",
    "3連複_人気",
]

results_df = results_df.drop(columns=drop_columns)
results_df.head()

In [ ]:
# レースタイム 分.秒.ms 形式を秒に変換する関数
def time_to_seconds(time_str):
    try:
        minutes, seconds, milli100seconds = time_str.split(".")
        total_seconds = int(minutes) * 60 + float(seconds) + float(milli100seconds) / 10
        return total_seconds
    except:
        return np.nan


# レースタイム列を秒に変換
results_df["レースタイム秒"] = results_df["レースタイム"].apply(time_to_seconds)
results_df.head()

In [ ]:
# レースタイム秒のヒストグラムをプロット
plt.figure(figsize=(10, 6))
sns.histplot(results_df["レースタイム秒"].dropna(), bins=50, kde=True)
plt.title("レースタイム秒の分布")
plt.xlabel("レースタイム秒")
plt.ylabel("頻度")
plt.show()

In [ ]:
# 距離が1800mのレースデータにフィルタリング
results_df = results_df[results_df["距離"] == 1800]
results_df.head()

In [ ]:
# 着順が "F", "S0", "S1", "S2", "L0", "L1", "K0", "K1" のデータはフィルタリング
results_df = results_df[
    ~results_df["着順"].isin(["F", "S0", "S1", "S2", "L0", "L1", "K0", "K1"])
]
# 着順を整数に変換
results_df["着順"] = results_df["着順"].astype(int)

In [ ]:
# スタートタイミングを数値に変換
results_df["スタートタイミング"] = pd.to_numeric(
    results_df["スタートタイミング"], errors="coerce"
)

In [ ]:
results_df.describe()

In [ ]:
# レースタイム秒のヒストグラムをプロット
plt.figure(figsize=(10, 6))
sns.histplot(results_df["レースタイム秒"].dropna(), bins=50, kde=True)
plt.title("レースタイム秒の分布 (距離1800m)")
plt.xlabel("レースタイム秒")
plt.ylabel("頻度")
plt.show()

In [ ]:
# レースID毎のレースタイム秒の平均,最小,最大を計算
average_times = (
    results_df.groupby("レースID")["レースタイム秒"]
    .agg(["mean", "min", "max"])
    .reset_index()
)
average_times.columns = [
    "レースID",
    "平均レースタイム秒",
    "最小レースタイム秒",
    "最大レースタイム秒",
]
average_times.describe()

In [ ]:
# average_timesの平均レースタイム秒のヒストグラムをプロット
plt.figure(figsize=(10, 6))
sns.histplot(average_times["平均レースタイム秒"].dropna(), bins=50, kde=True)
plt.title("レースID毎の平均レースタイム秒の分布")
plt.xlabel("平均レースタイム秒")
plt.ylabel("頻度")
plt.show()

In [ ]:
# 着順毎にデータを分離
results_1st_df = results_df[results_df["着順"] == 1]
results_2nd_df = results_df[results_df["着順"] == 2]
results_3rd_df = results_df[results_df["着順"] == 3]
results_4th_df = results_df[results_df["着順"] == 4]
results_5th_df = results_df[results_df["着順"] == 5]
results_6th_df = results_df[results_df["着順"] == 6]

In [ ]:
# 着順毎のレースタイム秒のヒストグラムをプロット
plt.figure(figsize=(12, 8))
sns.histplot(
    results_1st_df["レースタイム秒"].dropna(),
    bins=50,
    color="gold",
    label="1着",
    kde=True,
)
sns.histplot(
    results_2nd_df["レースタイム秒"].dropna(),
    bins=50,
    color="silver",
    label="2着",
    kde=True,
    alpha=0.5,
)
sns.histplot(
    results_3rd_df["レースタイム秒"].dropna(),
    bins=50,
    color="#cd7f32",
    label="3着",
    kde=True,
    alpha=0.5,
)
sns.histplot(
    results_4th_df["レースタイム秒"].dropna(),
    bins=50,
    color="blue",
    label="4着",
    kde=True,
    alpha=0.5,
)
sns.histplot(
    results_5th_df["レースタイム秒"].dropna(),
    bins=50,
    color="green",
    label="5着",
    kde=True,
    alpha=0.5,
)
sns.histplot(
    results_6th_df["レースタイム秒"].dropna(),
    bins=50,
    color="red",
    label="6着",
    kde=True,
    alpha=0.5,
)
plt.title("着順毎のレースタイム秒の分布")
plt.xlabel("レースタイム秒")
plt.ylabel("頻度")
plt.legend()
plt.show()

In [ ]:
# 選手登番毎の出走回数と1-6着回数と1-6着率を計算
place_stats_df = results_df.groupby("選手登番").agg(
    出走回数=("着順", "count"),
    着順1回数=("着順", lambda x: (x == 1).sum()),
    着順2回数=("着順", lambda x: (x == 2).sum()),
    着順3回数=("着順", lambda x: (x == 3).sum()),
    着順4回数=("着順", lambda x: (x == 4).sum()),
    着順5回数=("着順", lambda x: (x == 5).sum()),
    着順6回数=("着順", lambda x: (x == 6).sum()),
)
place_stats_df["着順1着率"] = place_stats_df["着順1回数"] / place_stats_df["出走回数"]
place_stats_df["着順2着率"] = place_stats_df["着順2回数"] / place_stats_df["出走回数"]
place_stats_df["着順3着率"] = place_stats_df["着順3回数"] / place_stats_df["出走回数"]
place_stats_df["着順4着率"] = place_stats_df["着順4回数"] / place_stats_df["出走回数"]
place_stats_df["着順5着率"] = place_stats_df["着順5回数"] / place_stats_df["出走回数"]
place_stats_df["着順6着率"] = place_stats_df["着順6回数"] / place_stats_df["出走回数"]
place_stats_df = place_stats_df.reset_index()
place_stats_df.head(10)

In [ ]:
# 選手登番毎の平均レースタイム秒と標準偏差, スタートタイミングとその標準偏差を計算
time_stats_df = results_df.groupby("選手登番").agg(
    平均レースタイム秒=("レースタイム秒", "mean"),
    レースタイム秒標準偏差=("レースタイム秒", "std"),
    平均スタートタイミング=("スタートタイミング", "mean"),
    スタートタイミング標準偏差=("スタートタイミング", "std"),
)
time_stats_df = time_stats_df.reset_index()
time_stats_df.head(10)

In [ ]:
# place_stats_dfとtime_stats_dfを選手登番で結合
player_stats_df = pd.merge(place_stats_df, time_stats_df, on="選手登番")
player_stats_df.head(10)

In [ ]:
len(player_stats_df)

In [ ]:
# 出走回数何回以上の選手を対象とするか
min_race_num = 100

In [ ]:
# 出走回数が100回以上の選手にフィルタリング
player_stats_df = player_stats_df[player_stats_df["出走回数"] >= min_race_num].copy()
player_stats_df = player_stats_df.reset_index(drop=True)
player_stats_df.describe()

In [ ]:
len(player_stats_df)

In [ ]:
# 着順1着率のヒストグラムをプロット
plt.figure(figsize=(10, 6))
sns.histplot(player_stats_df["着順1着率"].dropna(), bins=50, kde=True)
plt.title("選手登番毎の着順1着率の分布")
plt.xlabel("着順1着率")
plt.ylabel("頻度")
plt.show()

In [ ]:
# 平均レースタイム秒のヒストグラムをプロット
plt.figure(figsize=(10, 6))
sns.histplot(player_stats_df["平均レースタイム秒"].dropna(), bins=50, kde=True)
plt.title("選手登番毎の平均レースタイム秒の分布")
plt.xlabel("平均レースタイム秒")
plt.ylabel("頻度")
plt.show()

In [ ]:
# 上位、下位何名のデータを抽出するか
select_num = 200

In [ ]:
# 着順1着率上位選手の表示
top_players = player_stats_df.sort_values(by="着順1着率", ascending=False).head(
    select_num
)
top_players

In [ ]:
# 着順1着率下位選手の表示
bottom_players = player_stats_df.sort_values(by="着順1着率", ascending=True).head(
    select_num
)
bottom_players

In [ ]:
# top_100_playersとbottom_500_playersの平均レースタイム秒の箱ひげ図をプロット
plt.figure(figsize=(10, 6))
data_to_plot = [
    top_players["平均レースタイム秒"].dropna(),
    bottom_players["平均レースタイム秒"].dropna(),
]
plt.boxplot(data_to_plot, tick_labels=["着順1着率上位選手", "着順1着率下位選手"])
plt.title("平均レースタイム秒の比較")
plt.ylabel("平均レースタイム秒")
plt.show()

In [ ]:
# top_500_playersとbottom_500_playersのレースタイム秒標準偏差の箱ひげ図をプロット
plt.figure(figsize=(10, 6))
data_to_plot = [
    top_players["レースタイム秒標準偏差"].dropna(),
    bottom_players["レースタイム秒標準偏差"].dropna(),
]
plt.boxplot(data_to_plot, tick_labels=["着順1着率上位選手", "着順1着率下位選手"])
plt.title("レースタイム秒標準偏差の比較")
plt.ylabel("レースタイム秒標準偏差")
plt.show()

In [ ]:
# top_500_playersとbottom_500_playersの平均レースタイム秒のヒストグラムをプロット
plt.figure(figsize=(12, 6))
sns.histplot(
    top_players["平均レースタイム秒"].dropna(),
    bins=50,
    color="blue",
    label="着順1着率上位選手",
    kde=True,
)
sns.histplot(
    bottom_players["平均レースタイム秒"].dropna(),
    bins=50,
    color="red",
    label="着順1着率下位選手",
    kde=True,
    alpha=0.5,
)
plt.title("平均レースタイム秒の分布比較")
plt.xlabel("平均レースタイム秒")
plt.ylabel("頻度")
plt.legend()
plt.show()

In [ ]:
# top_500_playersとbottom_500_playersの平均レースタイム秒の密度をプロット
plt.figure(figsize=(12, 6))
sns.kdeplot(
    top_players["平均レースタイム秒"].dropna(),
    color="blue",
    label="着順1着率上位選手",
    fill=True,
    alpha=0.5,
)
sns.kdeplot(
    bottom_players["平均レースタイム秒"].dropna(),
    color="red",
    label="着順1着率下位選手",
    fill=True,
    alpha=0.5,
)
plt.title("平均レースタイム秒の密度比較")
plt.xlabel("平均レースタイム秒")
plt.ylabel("密度")
plt.legend()
plt.show()